In [1]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

In [ ]:
configuration = {
    "configurable":{
        "thread_id" : "test-2"
    }
}

In [3]:
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer= InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model= "groq:openai/gpt-oss-20b",
        trigger=("messages",10),
        keep = ("messages",4)
        
    )]
)

In [5]:
questions = [
    "Why the sky is blue?",
    "Why we do organisms live, what's their purpose?",
    "Why ADHD people often forget stuff immediately?",
    "Why ADHD guys can't be normal being?",
    "Why ADHD people learn things fast?",
    "A short description about Grinch"
]
for q in questions:
    response = agent.invoke({"messages": HumanMessage(content = q)},
                            config=configuration)
    print(f"Messages: {response}")
    print(f"No.of Messgaes: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT  \nThe user wants an explanation of why people with ADHD often struggle with everyday tasks and may feel “not normal.” They have previously asked about ADHD-related forgetfulness and are now asking for a broader explanation of the challenges faced by individuals with ADHD.\n\n## SUMMARY  \n**Key points from prior AI response on ADHD forgetfulness**  \n- **Working memory deficits**: Difficulty holding and manipulating information over short periods.  \n- **Executive function deficits**: Problems with planning, organization, and time management.  \n- **Distractibility**: Easily diverted attention from tasks.  \n- **Brain structure/function differences**: Especially in the prefrontal cortex, affecting working memory and executive control.  \n- **Environmental and personal factors**: Stress, anxiety, sleep deprivation, cluttered surroundings, lack of routine, and low motivation

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            trigger=("tokens",550),
            keep=("tokens",200)
            #trigger=("fraction", 0.005) - 0.005 or 5%  of maximum context window tokens
            #keep=("fraction", 0.002)
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token


In [10]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~347 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='08b3fb2e-17c6-40c3-bce5-b35eb33247c1'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in Paris. We have a function search_hotels that takes city. We should call it.', 'tool_calls': [{'id': 'fc_71788797-5ca2-4d7b-947e-f90bc9bb7a9f', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 129, 'total_tokens': 183, 'completion_time': 0.112571069, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.005692757, 'prompt_tokens_details': None, 'queue_time': 0.286743859, 'total_time': 0.118263826}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_3af2d41834', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f